In [ ]:
# Setup. Everything below runs the CLI itself, so the notebook and the PBS job
# execute identical code.
import logging
import os
import sys

for _candidate in [".", "..", "../.."]:  # noqa: B007
    _p = os.path.abspath(_candidate)
    if os.path.isdir(os.path.join(_p, "src")):
        sys.path.insert(0, _p)
        break

from IPython.display import Image, Markdown, display  # noqa: E402

from scripts.run_corrca_condition_tracks import _build_arg_parser, run  # noqa: E402
from src.definitions.constants import ProjectPaths  # noqa: E402

logging.basicConfig(
    level=logging.INFO, format="%(levelname)s %(name)s: %(message)s", force=True
)
print("Setup complete.")

# Stage 06b — CorrCA across conditions (joined tracks)

Runs CorrCA on each participant's Placebo track followed by their Psilocybin track: one shared channel filter per component, fitted over both conditions, then the projected sources split back into the two segments. The synchrony report gives each component's ISC within each condition, through one fixed filter.

**The method is documented in
[`docs/analyses/06b-corrca-condition-comparison/`](../../docs/analyses/06b-corrca-condition-comparison/README.md)** — this
notebook does not restate it.

**What this notebook is for.** Choosing parameters and checking the workflow cheaply
before committing a node to it. It calls `run()` from
[`scripts/run_corrca_condition_tracks.py`](../../scripts/run_corrca_condition_tracks.py) — the same function the PBS job
calls — at reduced scale, then shows what it wrote.

## Configuration

In [ ]:
# The decomposition to run. Small on purpose -- the full-size run is the job's.
EXPERIMENT = "assr"
N_PCA = 10  # pooled covariance subspace = number of components
GAMMA = 0.0  # covariance shrinkage; 0 relies on the N_PCA truncation
# None = broadband. (30, 50) = the gamma-band run of the *_gamma.pbs jobs: the
# frequency axis is cut BEFORE the fit, and figures land under bands/ as 30-50hz_*.
BAND_RANGE = None
ZSCORE_MODE = "per_condition"  # or "joint" to keep the power contrast

# Every run also fits on the binary ASSR ROI alone (cut BEFORE the z-scoring),
# stored and drawn as the '_roi' variant. A leading channel slice holds only a
# few ASSR electrodes, so with N_CHANNELS set that fit is small (or skipped when
# it holds fewer than N_PCA); N_CHANNELS = None gives the job's ROI.
SKIP_ROI_FIT = False

# Subset every axis that can be subset. Channels and pairs are what make this cheap;
# the wavelet transform itself is reused from the cache rather than recomputed.
N_PAIRS = 6  # participants
N_CHANNELS = 32  # None = all; the ASSR mask needs --lenient_mask at 32
N_TIMES = 3000  # samples per condition (~12 s at 250 Hz)

OUT_DIR = (
    ProjectPaths.NOTEBOOKS_DIR
    / "06b-corrca-condition-comparison"
    / "out"
    / "joined_tracks"
)
PLOTS_DIR = OUT_DIR / "plots"

## Run

The same function the PBS job calls. Keep the scale small here: the full-size pass is what
[`jobs/metacentrum/06b-corrca-condition-comparison/`](../../jobs/metacentrum/06b-corrca-condition-comparison) is for.
The per-condition ISC of every component is logged; the tables land beside the figures.

In [ ]:
args = _build_arg_parser().parse_args(
    [
        "--experiment",
        EXPERIMENT,
        "--n_pca",
        str(N_PCA),
        "--gamma",
        str(GAMMA),
        "--zscore_mode",
        ZSCORE_MODE,
        *(["--band_range", *map(str, BAND_RANGE)] if BAND_RANGE else []),
        "--n_pairs",
        str(N_PAIRS),
        *(["--n_channels", str(N_CHANNELS)] if N_CHANNELS else []),
        *(["--skip_roi_fit"] if SKIP_ROI_FIT else []),
        "--n_times",
        str(N_TIMES),
        "--reuse_wavelets",
        "--lenient_mask",  # a 32-channel subset drops most ASSR electrodes
        "--save_dir",
        str(PLOTS_DIR),
    ]
)
run(args)

## Figures

The condition grids are drawn by the shared plotters in
[`src/visualization/iva_condition_plots.py`](../../src/visualization/iva_condition_plots.py),
the synchrony report by
[`src/visualization/corrca_plots.py`](../../src/visualization/corrca_plots.py) -- both
used by the CLI too, so what is shown here is what the job produces.

In [ ]:
import pandas as pd  # noqa: E402

for csv in sorted(PLOTS_DIR.rglob("corrca_*.csv")):
    display(Markdown(f"**{csv.relative_to(PLOTS_DIR)}**"))
    display(pd.read_csv(csv))
for png in sorted(PLOTS_DIR.rglob("*.png")):
    display(Markdown(f"**{png.relative_to(PLOTS_DIR)}**"))
    display(Image(filename=str(png)))